In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, VBox, HBox, HTML, Output, Layout
from IPython.display import display, clear_output

# ============================================================
# UNIFORM VS FLOATING-POINT QUANTIZATION
# ============================================================

# ============================================================
# SHORT DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.42;
    width:550px;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#4f5f9b;
    margin-bottom:8px;
">
Uniform and Floating-Point Quantizers
</div>

<div style="margin-bottom:5px;">
A uniform fixed-point quantizer uses the same spacing Δ between all successive output levels.
</div>

<div style="margin-bottom:5px;">
In a floating-point quantizer, the number of representable values between successive powers of two is approximately constant, so the spacing between levels increases with signal magnitude.
</div>

<div style="margin-bottom:5px;">
Consequently, floating-point quantization is nonuniform: small amplitudes are represented with finer absolute resolution than large amplitudes.
</div>

<div>
<b>This notebook:</b> compares the input/output characteristics and the local quantization-step size of the two quantizers.
</div>

</div>
""")

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '0px'}

slider_layout = Layout(
    width='125px',
    min_width='125px'
)

delta_slider = FloatSlider(
    min=0.10,
    max=2.00,
    step=0.10,
    value=0.50,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

p_slider = IntSlider(
    min=2,
    max=10,
    step=1,
    value=4,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

xmax_slider = FloatSlider(
    min=4.0,
    max=32.0,
    step=2.0,
    value=16.0,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

# ============================================================
# VALUE LABELS
# ============================================================

value_layout = Layout(
    width='55px',
    min_width='55px',
    margin='0px 0px 0px 4px'
)

delta_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.50</div>',
    layout=value_layout
)

p_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">4</div>',
    layout=value_layout
)

xmax_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">16</div>',
    layout=value_layout
)

# ============================================================
# LABELS
# ============================================================

label_layout = Layout(
    width='155px',
    min_width='155px'
)

delta_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Uniform step Δ:</div>',
    layout=label_layout
)

p_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Significand bits p:</div>',
    layout=label_layout
)

xmax_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Display range ±X:</div>',
    layout=label_layout
)

# ============================================================
# CONTROL ROWS
# ============================================================

row_layout = Layout(
    width='390px',
    min_width='390px',
    height='38px',
    min_height='38px',
    align_items='center',
    overflow='visible'
)

delta_row = HBox(
    [
        delta_label,
        delta_slider,
        delta_value
    ],
    layout=row_layout
)

p_row = HBox(
    [
        p_label,
        p_slider,
        p_value
    ],
    layout=row_layout
)

xmax_row = HBox(
    [
        xmax_label,
        xmax_slider,
        xmax_value
    ],
    layout=row_layout
)

# ============================================================
# CONTROLS CARD
# ============================================================

controls_card = VBox(
    [
        HTML("""
        <div style="
            font-family:Arial;
            font-size:17px;
            font-weight:bold;
            color:#4f5f9b;
            margin-bottom:8px;
        ">
        Quantizer Parameters
        </div>
        """),

        delta_row,
        p_row,
        xmax_row
    ],
    layout=Layout(
        width='420px',
        min_width='420px',
        padding='10px 12px 12px 12px',
        border='1px solid #c3cae2',
        overflow='visible'
    )
)

# ============================================================
# TOP TWO-COLUMN LAYOUT
# ============================================================

top_layout = HBox(
    [
        documentation,
        controls_card
    ],
    layout=Layout(
        width='1020px',
        align_items='flex-start',
        justify_content='space-between',
        gap='16px',
        margin='0px 0px 10px 0px',
        overflow='visible'
    )
)

# ============================================================
# OUTPUT AREAS
# ============================================================

graph_output = Output(
    layout=Layout(
        width='1080px',
        overflow='hidden'
    )
)

result_html = HTML()

# ============================================================
# UNIFORM QUANTIZER
# ============================================================

def uniform_quantize(x, delta):

    return delta * np.floor(
        x / delta + 0.5
    )

# ============================================================
# FLOATING-POINT QUANTIZER
#
# p = number of significant binary bits.
#
# For
#
# 2^k <= |x| < 2^(k+1)
#
# the local spacing is
#
# D = 2^(k-p+1)
#
# ============================================================

def floating_quantize(x, p):

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.zeros_like(
        x
    )

    nonzero = (
        x != 0
    )

    magnitude = np.abs(
        x[nonzero]
    )

    exponent = np.floor(
        np.log2(magnitude)
    ).astype(int)

    step = 2.0 ** (
        exponent - p + 1
    )

    y[nonzero] = np.sign(
        x[nonzero]
    ) * np.round(
        magnitude / step
    ) * step

    return y

# ============================================================
# LOCAL FLOATING-POINT STEP
# ============================================================

def floating_step(x, p):

    x = np.asarray(
        x,
        dtype=float
    )

    step = np.zeros_like(
        x
    )

    nonzero = (
        x != 0
    )

    exponent = np.floor(
        np.log2(np.abs(x[nonzero]))
    ).astype(int)

    step[nonzero] = 2.0 ** (
        exponent - p + 1
    )

    return step

# ============================================================
# MAIN PLOT FUNCTION
# ============================================================

def plot_quantizers(delta, p, xmax):

    # --------------------------------------------------------
    # INPUT AXIS
    # --------------------------------------------------------

    x = np.linspace(
        -xmax,
        xmax,
        5000
    )

    # --------------------------------------------------------
    # QUANTIZED OUTPUTS
    # --------------------------------------------------------

    y_uniform = uniform_quantize(
        x,
        delta
    )

    y_float = floating_quantize(
        x,
        p
    )

    # --------------------------------------------------------
    # LOCAL FLOATING-POINT STEP
    # --------------------------------------------------------

    x_positive = np.logspace(
        -3,
        np.log10(xmax),
        1800
    )

    float_step = floating_step(
        x_positive,
        p
    )

    # ========================================================
    # FIGURE
    # ========================================================

    fig = plt.figure(
        figsize=(10.6, 7.0)
    )

    gs = fig.add_gridspec(
        2,
        2,
        height_ratios=[1.0, 0.92],
        hspace=0.46,
        wspace=0.30
    )

    ax1 = fig.add_subplot(
        gs[0, 0]
    )

    ax2 = fig.add_subplot(
        gs[0, 1]
    )

    ax3 = fig.add_subplot(
        gs[1, :]
    )

    # ========================================================
    # GRAPH 1:
    # UNIFORM QUANTIZER
    # ========================================================

    ax1.plot(
        x,
        y_uniform,
        linewidth=1.4,
        label='Uniform quantizer'
    )

    ax1.plot(
        x,
        x,
        linestyle='--',
        linewidth=1.0,
        label='y = x'
    )

    ax1.set_xlim(
        -xmax,
        xmax
    )

    ax1.set_ylim(
        -xmax,
        xmax
    )

    ax1.set_xlabel(
        'Input x',
        fontsize=10
    )

    ax1.set_ylabel(
        'Quantized output',
        fontsize=10
    )

    ax1.set_title(
        'Uniform Fixed-Point Quantizer',
        fontsize=12,
        pad=8
    )

    ax1.tick_params(
        axis='both',
        labelsize=9
    )

    ax1.grid(
        True,
        linestyle=':',
        alpha=0.4
    )

    ax1.legend(
        loc='upper left',
        fontsize=8
    )

    # ========================================================
    # GRAPH 2:
    # FLOATING-POINT QUANTIZER
    # ========================================================

    ax2.plot(
        x,
        y_float,
        linewidth=1.4,
        label='Floating-point quantizer'
    )

    ax2.plot(
        x,
        x,
        linestyle='--',
        linewidth=1.0,
        label='y = x'
    )

    ax2.set_xlim(
        -xmax,
        xmax
    )

    ax2.set_ylim(
        -xmax,
        xmax
    )

    ax2.set_xlabel(
        'Input x',
        fontsize=10
    )

    ax2.set_ylabel(
        'Quantized output',
        fontsize=10
    )

    ax2.set_title(
        f'Floating-Point Quantizer, p = {p}',
        fontsize=12,
        pad=8
    )

    ax2.tick_params(
        axis='both',
        labelsize=9
    )

    ax2.grid(
        True,
        linestyle=':',
        alpha=0.4
    )

    ax2.legend(
        loc='upper left',
        fontsize=8
    )

    # ========================================================
    # GRAPH 3:
    # LOCAL STEP SIZE
    # ========================================================

    ax3.semilogx(
        x_positive,
        float_step,
        linewidth=2.0,
        label='Floating-point local step'
    )

    ax3.axhline(
        delta,
        linestyle='--',
        linewidth=1.5,
        label='Uniform step Δ'
    )

    ax3.set_xlim(
        1e-3,
        xmax
    )

    ax3.set_ylim(
        0,
        max(
            1.10 * delta,
            1.15 * np.max(float_step)
        )
    )

    ax3.set_xlabel(
        'Signal magnitude |x|',
        fontsize=10
    )

    ax3.set_ylabel(
        'Distance between adjacent levels',
        fontsize=10
    )

    ax3.set_title(
        'Local Quantization-Step Size',
        fontsize=12,
        pad=8
    )

    ax3.tick_params(
        axis='both',
        labelsize=9
    )

    ax3.grid(
        True,
        which='major',
        linestyle=':',
        alpha=0.4
    )

    ax3.legend(
        loc='upper left',
        fontsize=8
    )

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    fig.subplots_adjust(
        left=0.08,
        right=0.97,
        top=0.93,
        bottom=0.10
    )

    plt.show()

    plt.close(fig)

    # ========================================================
    # NUMERICAL VALUES
    # ========================================================

    test_values = np.array(
        [
            1.0,
            2.0,
            4.0,
            8.0
        ]
    )

    test_values = test_values[
        test_values <= xmax
    ]

    test_steps = floating_step(
        test_values,
        p
    )

    if len(test_values) > 0:

        step_text = ' &nbsp;&nbsp; '.join(
            [
                f'D(|x|={value:g})={step:.5f}'
                for value, step in zip(
                    test_values,
                    test_steps
                )
            ]
        )

    else:

        step_text = ''

    # ========================================================
    # RESULT BOX
    # ========================================================

    result_html.value = f"""
    <div style="
        font-family:Arial, sans-serif;
        font-size:15px;
        line-height:1.50;
        width:980px;
        padding:10px 14px;
        border:1px solid #d7c38d;
        background:#fffbed;
        box-sizing:border-box;
    ">

    <b>Uniform quantizer:</b>
    Δ = {delta:.4f} at every amplitude.

    <br>

    <b>Floating-point quantizer:</b>
    p = {p} significant bits.

    <br>

    <b>Examples of local floating-point spacing:</b>
    {step_text}

    </div>
    """

# ============================================================
# UPDATE FUNCTION
# ============================================================

def update_notebook(change=None):

    # --------------------------------------------------------
    # CURRENT VALUES
    # --------------------------------------------------------

    delta_value.value = f"""
    <div style="
        font-family:Arial;
        font-size:14px;
        font-weight:bold;
        color:#0b3d91;
    ">
    {delta_slider.value:.2f}
    </div>
    """

    p_value.value = f"""
    <div style="
        font-family:Arial;
        font-size:14px;
        font-weight:bold;
        color:#0b3d91;
    ">
    {p_slider.value}
    </div>
    """

    xmax_value.value = f"""
    <div style="
        font-family:Arial;
        font-size:14px;
        font-weight:bold;
        color:#0b3d91;
    ">
    {xmax_slider.value:.0f}
    </div>
    """

    # --------------------------------------------------------
    # REDRAW
    # --------------------------------------------------------

    with graph_output:

        clear_output(
            wait=True
        )

        plot_quantizers(
            delta_slider.value,
            p_slider.value,
            xmax_slider.value
        )

# ============================================================
# CONNECT CONTROLS
# ============================================================

delta_slider.observe(
    update_notebook,
    names='value'
)

p_slider.observe(
    update_notebook,
    names='value'
)

xmax_slider.observe(
    update_notebook,
    names='value'
)

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.45;
    width:1080px;
    padding:11px 15px;
    border:1px solid #c8cee5;
    background:#f8f9fe;
    box-sizing:border-box;
    margin-top:6px;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#4f5f9b;
    margin-bottom:6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:4px;">
The uniform quantizer has the same absolute resolution Δ everywhere, so its input/output staircase has equally spaced levels.
</div>

<div style="margin-bottom:4px;">
The floating-point quantizer uses progressively larger level spacing as |x| crosses successive powers of two; therefore its quantization is intrinsically nonuniform.
</div>

<div>
Increasing the number of significand bits p reduces the spacing within every exponent range and improves the floating-point resolution without removing its amplitude dependence.
</div>

</div>
""")

# ============================================================
# INITIAL DRAW
# ============================================================

update_notebook()

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        top_layout,
        graph_output,
        result_html,
        interpretation
    ],
    layout=Layout(
        width='1080px',
        overflow='visible'
    )
)

display(main_layout)